# Security ML Threat Model

## What This Is
Machine learning for security has distinct failure modes from general ML. A security ML threat model must account not just for distribution shift but for *adversarial adaptation* — attackers who observe your model and modify their behavior to evade it.

## What ML Can Do for Security
- **Anomaly detection at scale**: Process millions of log events, network flows, or file executions
- **Pattern recognition across large feature spaces**: Malware with 50,000 PE features, network flows with 80 features
- **Behavioral baselining**: Learn normal patterns and flag deviations (UEBA)
- **Classification with weak labels**: Alert triage, incident severity scoring

## What ML Can't Do
- **Handle adversarial evasion robustly without formal guarantees**: Attackers adapt; your model's accuracy degrades
- **Detect zero-days without anomaly signals**: If a 0-day perfectly mimics legitimate traffic, supervised ML won't catch it
- **Replace analyst judgment on ambiguous alerts**: False positive rates matter hugely — a 1% FPR at 1M events/day = 10,000 false alarms
- **Provide attribution**: Classification ≠ attribution

## The Adversarial ML Loop
Security ML operates in a red team / blue team feedback loop: attackers observe ML-based defenses and craft evasion strategies. This is fundamentally different from benign distribution shift.

In [ ]:
import numpy as np
from typing import Dict, List, Tuple

np.random.seed(42)

# Security ML capability audit
# Model performance at different operating points matters differently for security

class SecurityMLAudit:
    """Evaluate ML model fitness for security use cases."""
    
    def __init__(self, tpr: float, fpr: float, events_per_day: int = 1_000_000):
        self.tpr = tpr  # True Positive Rate (sensitivity)
        self.fpr = fpr  # False Positive Rate
        self.epd = events_per_day
    
    def analyst_load(self, base_rate: float) -> Dict:
        """Compute analyst alert volume at a given attack base rate."""
        true_attacks = self.epd * base_rate
        true_negatives = self.epd * (1 - base_rate)
        
        tp = true_attacks * self.tpr
        fp = true_negatives * self.fpr
        total_alerts = tp + fp
        precision = tp / (tp + fp) if tp + fp > 0 else 0
        
        return {
            'base_rate': base_rate,
            'alerts_per_day': int(total_alerts),
            'true_positives': int(tp),
            'false_positives': int(fp),
            'precision': precision,
            'analyst_hours_8h_day': total_alerts / 20  # ~3 min per alert
        }
    
    def print_report(self):
        print(f'Security ML Capability Audit')
        print(f'Model: TPR={self.tpr:.1%}, FPR={self.fpr:.3%}')
        print(f'Volume: {self.epd:,} events/day')
        print()
        print(f'{'Base Rate':<12} {'Alerts/Day':>11} {'Precision':>10} {'FTE Analysts':>13}')
        print('-' * 50)
        for br in [0.0001, 0.001, 0.01, 0.05]:
            r = self.analyst_load(br)
            fte = r['analyst_hours_8h_day'] / 8
            print(f'{br:<12.4f} {r["alerts_per_day"]:>11,} {r["precision"]:>10.3f} {fte:>13.1f}')

# Good model: 95% TPR, 0.1% FPR
print('=== High-quality model ===')
audit_good = SecurityMLAudit(tpr=0.95, fpr=0.001)
audit_good.print_report()

print('\n=== Weak model (98% TPR, 5% FPR) ===')
audit_weak = SecurityMLAudit(tpr=0.98, fpr=0.05)
audit_weak.print_report()


In [ ]:
# Adversarial evasion threat model
# How quickly can attackers evade an ML-based detector?

class AdversarialEvolutionSimulator:
    """Simulate attacker adaptation to an ML-based detector over time."""
    
    def __init__(self, initial_detection_rate: float = 0.92):
        self.detection_rate = initial_detection_rate
        self.rounds = []
    
    def simulate_round(self, retrain: bool = False):
        """Simulate one round of attacker adaptation."""
        # Without retraining: attackers learn to evade; detection rate drops
        if not retrain:
            # Attackers observe FPs/FNs and tune their tools
            evasion_gain = np.random.uniform(0.03, 0.08)  # 3-8% evasion gain per round
            self.detection_rate = max(0.1, self.detection_rate - evasion_gain)
        else:
            # Defenders retrain on new samples; detection recovers partially
            recovery = np.random.uniform(0.04, 0.10)
            self.detection_rate = min(0.95, self.detection_rate + recovery)
        
        self.rounds.append(self.detection_rate)
        return self.detection_rate

# No retraining scenario
sim_no_retrain = AdversarialEvolutionSimulator(0.92)
dr_no_retrain = [0.92]
for i in range(10):
    dr_no_retrain.append(sim_no_retrain.simulate_round(retrain=False))

# With periodic retraining
sim_retrain = AdversarialEvolutionSimulator(0.92)
dr_retrain = [0.92]
for i in range(10):
    retrain_this_round = (i % 3 == 2)  # retrain every 3 rounds
    dr_retrain.append(sim_retrain.simulate_round(retrain=retrain_this_round))

print('Adversarial Evolution Simulation')
print('Round | No Retrain | Periodic Retrain')
print('-' * 40)
for i in range(11):
    marker = ' <retrain' if i > 0 and i % 3 == 0 else ''
    print(f'  {i:2d}  |   {dr_no_retrain[i]:.3f}    |     {dr_retrain[i]:.3f}{marker}')

print('\nKey insight: static ML models in security decay rapidly under adversarial pressure.')
print('Continuous learning pipelines (Series 27) are essential for security applications.')
